#### Transform Refunds Tables
- Extract specific portion of string from refund_reason using split function
- Extract specific portion of string from refund_reason using regexp_extract function
- Extract the date and time from refund_timestamp
- Write transformed data into the silver layer

#### 0. Create the bronze refunds table from Azure SQL

In [0]:
df = spark.read.table("gizmobox.bronze.py_refunds")
display(df)

#### 1. Extract specific portion of string from refund_reason using split function

https://docs.databricks.com/aws/en/sql/language-manual/functions/split



In [0]:
from pyspark.sql import functions as F

extract_df = df.select(
    "refund_id",
    "payment_id",
    F.to_date("refund_timestamp").alias("refund_date"),
    "refund_amount",
    F.split("refund_reason", ":")[0].alias("refund_reason"),
    F.split("refund_reason", ":")[1].alias("refund_reason_code")
)

display(extract_df)


In [0]:
%sql
SELECT 
    *,
    split(refund_reason, ':')[0] AS refund_reason,
    split(refund_reason, ':')[1] AS refund_source
FROM gizmobox.bronze.refunds

#### 2. Extract specific portion of string from refund_reason using regexp_extract function
- 
- https://docs.databricks.com/aws/en/sql/language-manual/functions/regexp_extract
- https://regexr.com/
- https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_extract.html

In [0]:
from pyspark.sql import functions as F

extract_df = df.select(
    "refund_id",
    "payment_id",
    F.to_date("refund_timestamp").alias("refund_date"),
    "refund_amount",
    F.regexp_extract("refund_reason", "^([^:]+):", 1).alias("refund_reason"),
    F.regexp_extract("refund_reason", "([^:]+)$", 1).alias("refund_reason_code")
)

display(extract_df)


#### 3. Extract the date and time from refund_timestamp

In [0]:
from pyspark.sql import functions as F

extract_df = df.select(
    "refund_id",
    "payment_id",
    F.to_date("refund_timestamp").alias("refund_date"),
    "refund_amount",
    F.regexp_extract("refund_reason", "^([^:]+):", 1).alias("refund_reason"),
    F.regexp_extract("refund_reason", "([^:]+)$", 1).alias("refund_reason_code")
)

display(extract_df)

#### 4. Write transformed data into the silver layer

In [0]:
extract_df.writeTo("gizmobox.silver.py_refunds").createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_refunds